In [ ]:
import time
from io import BytesIO
from pathlib import Path
from random import Random

import pandas as pd
import requests
from PIL import Image
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


API = "https://collectionapi.metmuseum.org/public/collection/v1"
TARGETS = {17: 67, 18: 66, 19: 67}

dataset_folder = Path("met_sample").resolve()
image_folder = dataset_folder / "images"
image_folder.mkdir(parents=True, exist_ok=True)

metadata_path = dataset_folder / "metadata_balanced.csv"
original_csv = dataset_folder / "metadata.csv"

COLUMNS = [
    "object_id", "filename", "title", "artist", "date", "century",
    "medium", "classification", "image_url", "artwork_url",
]


def get_century(begin, end):
    if not isinstance(begin, int) or not isinstance(end, int):
        return None
    if not 1601 <= begin <= end <= 1900:
        return None

    first_century = (begin - 1) // 100 + 1
    last_century = (end - 1) // 100 + 1

    if first_century == last_century:
        return first_century
    return None


def image_is_usable(path):
    try:
        with Image.open(path) as image:
            image.load()
    except (OSError, ValueError):
        return False
    return True


def save_metadata(rows):
    temporary_path = metadata_path.with_suffix(".tmp")
    pd.DataFrame(rows, columns=COLUMNS).to_csv(temporary_path, index=False)
    temporary_path.replace(metadata_path)


rows = []
saved_ids = set()
counts = dict.fromkeys(TARGETS, 0)

# Resume from the balanced CSV and check that each saved image still opens.
if metadata_path.exists():
    for row in pd.read_csv(metadata_path).to_dict("records"):
        object_id = int(row["object_id"])
        century = int(row["century"])

        if (
            century in TARGETS
            and counts[century] < TARGETS[century]
            and object_id not in saved_ids
            and image_is_usable(dataset_folder / row["filename"])
        ):
            rows.append(row)
            saved_ids.add(object_id)
            counts[century] += 1

print(f"Starting with {len(rows)} saved paintings. Counts: {counts}")

# Check paintings from the original CSV first if it is still available.
original_ids = set()
if original_csv.exists():
    original_ids = set(pd.read_csv(original_csv)["object_id"].astype(int))

retry = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)

with requests.Session() as session:
    session.mount("https://", HTTPAdapter(max_retries=retry))

    try:
        if counts != TARGETS:
            response = session.get(
                f"{API}/objects",
                params={"departmentIds": 11},  # European Paintings
                timeout=30,
            )
            response.raise_for_status()
            object_ids = response.json().get("objectIDs") or []

            Random(42).shuffle(object_ids)
            object_ids.sort(key=lambda object_id: object_id not in original_ids)

            consecutive_metadata_403s = 0

            for object_id in object_ids:
                if counts == TARGETS:
                    break
                if object_id in saved_ids:
                    continue
                time.sleep(3)
                request_type = "metadata"

                try:
                    object_url = f"{API}/objects/{object_id}"
                    response = session.get(object_url, timeout=30)

                    if response.status_code == 403:
                        print(
                            f"Metadata request for {object_id} returned 403. "
                            "Waiting 60 seconds before retrying.",
                            flush=True,
                        )
                        time.sleep(60)
                        response = session.get(object_url, timeout=30)

                    response.raise_for_status()
                    consecutive_metadata_403s = 0
                    artwork = response.json()

                    century = get_century(
                        artwork.get("objectBeginDate"),
                        artwork.get("objectEndDate"),
                    )
                    image_url = artwork.get("primaryImageSmall")

                    if (
                        century not in TARGETS
                        or counts[century] >= TARGETS[century]
                        or artwork.get("classification") != "Paintings"
                        or not artwork.get("isPublicDomain")
                        or not image_url
                    ):
                        continue

                    filename = f"images/{object_id}.jpg"
                    image_path = dataset_folder / filename

                    if not image_is_usable(image_path):
                        request_type = "image"
                        response = session.get(image_url, timeout=30)
                        response.raise_for_status()

                        with Image.open(BytesIO(response.content)) as image:
                            image.convert("RGB").save(image_path)

                        if not image_is_usable(image_path):
                            print(f"Skipped {object_id}: saved image could not be read.")
                            continue

                except requests.HTTPError as error:
                    status = error.response.status_code
                    failed_url = error.response.url
                    print(
                        f"Skipped {object_id}: HTTP {status} "
                        f"from {request_type} request ({failed_url})"
                    )

                    if request_type == "metadata" and status == 403:
                        consecutive_metadata_403s += 1
                        if consecutive_metadata_403s >= 3:
                            print("The metadata API returned three consecutive 403s.")
                            break
                    else:
                        consecutive_metadata_403s = 0

                    continue

                except (requests.RequestException, OSError, ValueError) as error:
                    consecutive_metadata_403s = 0
                    print(f"Skipped {object_id} during {request_type} processing: {error}")
                    continue

                rows.append({
                    "object_id": object_id,
                    "filename": filename,
                    "title": artwork.get("title"),
                    "artist": artwork.get("artistDisplayName"),
                    "date": artwork.get("objectDate"),
                    "century": century,
                    "medium": artwork.get("medium"),
                    "classification": artwork.get("classification"),
                    "image_url": image_url,
                    "artwork_url": artwork.get("objectURL"),
                })
                saved_ids.add(object_id)
                counts[century] += 1
                save_metadata(rows)

                if len(rows) % 25 == 0:
                    print(f"Saved {len(rows)} paintings. Counts: {counts}")

    except requests.RequestException as error:
        print(f"Could not retrieve the Met object list: {error}")

    finally:
        save_metadata(rows)
        print(f"\nTotal saved: {len(rows)}")
        for century, target in TARGETS.items():
            print(f"{century}th century: {counts[century]}/{target}")

Starting with 58 saved paintings. Counts: {17: 20, 18: 12, 19: 26}
Saved 75 paintings. Counts: {17: 23, 18: 20, 19: 32}
Saved 100 paintings. Counts: {17: 32, 18: 25, 19: 43}
Saved 125 paintings. Counts: {17: 40, 18: 33, 19: 52}
Saved 150 paintings. Counts: {17: 50, 18: 36, 19: 64}


Saved 175 paintings. Counts: {17: 63, 18: 45, 19: 67}
Saved 200 paintings. Counts: {17: 67, 18: 66, 19: 67}

Total saved: 200
17th century: 67/67
18th century: 66/66
19th century: 67/67


In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

from google.colab import files


metadata = pd.read_csv(metadata_path)
image_data_path = str(image_folder)

if metadata.empty:
    raise ValueError("No paintings were saved. Check the output from Cell 1.")

actual_counts = metadata["century"].value_counts().to_dict()

for century, target in TARGETS.items():
    print(f"{century}th century: {actual_counts.get(century, 0)}/{target}")

if actual_counts != TARGETS:
    print("This ZIP contains a partial dataset. Rerun Cell 1 to continue collecting.")

zip_path = Path("met_sample_balanced.zip")

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(metadata_path, arcname="metadata_balanced.csv")

    for filename in metadata["filename"]:
        archive.write(dataset_folder / filename, arcname=filename)

print(f"Paintings in ZIP: {len(metadata)}")
files.download(str(zip_path))

17th century: 67/67
18th century: 66/66
19th century: 67/67
Paintings in ZIP: 200


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>